# Stock Price Prediction

Goal: pick up to 4000 stocks to sell and maximise R = sum of (p40 - p50) for the ones we choose.

Plan: predict p50 using gradient boosting models, then rank stocks by expected gain and sell the top 4000.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.ensemble import HistGradientBoostingRegressor

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

PRICE_COLS = [f'p{i}' for i in range(1, 41)]
MAX_SELL = 4000

## Load Data

In [ ]:
train = pd.read_csv('train.csv', index_col='ID')
test  = pd.read_csv('test.csv',  index_col='ID')

print(f'Train: {train.shape}  |  Test: {test.shape}')
train.head(3)

## EDA — Understanding the Target

We want to sell stocks where p40 > p50 (price drops over the next 10 periods).  
Let's see how many stocks that covers and how large the gains are.

In [ ]:
gain = train['p40'] - train['p50']

print(gain.describe().round(2))
print(f'\nStocks where p40 > p50: {(gain > 0).sum()} / {len(gain)} ({(gain > 0).mean():.1%})')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(gain, bins=80, color='steelblue', alpha=0.8)
axes[0].axvline(0, color='red', lw=1.5, ls='--', label='breakeven')
axes[0].set_title('Distribution of (p40 - p50)')
axes[0].set_xlabel('Gain from selling')
axes[0].legend()

axes[1].hist(train['p40'], bins=60, color='orange', alpha=0.7, label='p40')
axes[1].hist(train['p50'], bins=60, color='purple', alpha=0.6, label='p50')
axes[1].set_title('p40 vs p50 price distribution')
axes[1].legend()

plt.tight_layout()
plt.show()

## Feature Engineering

We build 33 features from the p1–p40 price history.

The features fall into a few groups:
- **Trend / momentum** — where is the price heading? (log returns, moving averages)
- **Volatility** — how noisy is the path? (standard deviation of price and returns)
- **Position** — where does p40 sit relative to its own history? (Bollinger band position, % of range)
- **RSI** — standard momentum indicator used in finance
- **Trend slope** — fitted slope of the log-price path

In [ ]:
def compute_rsi(prices, period=14):
    deltas = np.diff(prices[-period - 1:])
    gain   = np.mean(deltas[deltas > 0]) if np.any(deltas > 0) else 0.0
    loss   = np.mean(-deltas[deltas < 0]) if np.any(deltas < 0) else 0.0
    if loss == 0:
        return 100.0
    return 100.0 - 100.0 / (1.0 + gain / loss)


def engineer_features(df):
    p = np.clip(df[PRICE_COLS].values, 1e-6, None)
    n = len(df)
    f = {}

    f['p40'] = p[:, -1]
    f['p1']  = p[:, 0]

    f['log_ret_full']    = np.log(p[:, -1] / p[:, 0])
    f['last_change']     = p[:, -1] - p[:, -2]
    f['last_change_pct'] = f['last_change'] / p[:, -2]

    for w in [5, 10, 20]:
        f[f'ret_{w}d'] = np.log(p[:, -1] / p[:, -1 - w])

    f['ma5']  = p[:, -5:].mean(axis=1)
    f['ma10'] = p[:, -10:].mean(axis=1)
    f['ma20'] = p[:, -20:].mean(axis=1)
    f['ma40'] = p.mean(axis=1)

    f['p40_vs_ma5']  = f['p40'] - f['ma5']
    f['p40_vs_ma20'] = f['p40'] - f['ma20']
    f['p40_vs_ma40'] = f['p40'] - f['ma40']

    f['std10'] = p[:, -10:].std(axis=1)
    f['std20'] = p[:, -20:].std(axis=1)
    f['std40'] = p.std(axis=1)

    log_rets = np.diff(np.log(p), axis=1)
    f['vol10'] = log_rets[:, -10:].std(axis=1)
    f['vol20'] = log_rets[:, -20:].std(axis=1)
    f['vol40'] = log_rets.std(axis=1)

    upper = f['ma20'] + 2 * f['std20']
    lower = f['ma20'] - 2 * f['std20']
    band  = upper - lower
    f['bb_pos'] = np.where(band > 0, (f['p40'] - lower) / band, 0.5)

    f['hist_min']   = p.min(axis=1)
    f['hist_max']   = p.max(axis=1)
    f['hist_range'] = f['hist_max'] - f['hist_min']
    f['p40_vs_min'] = f['p40'] - f['hist_min']
    f['p40_vs_max'] = f['hist_max'] - f['p40']
    f['p40_pct_range'] = np.where(
        f['hist_range'] > 0, f['p40_vs_min'] / f['hist_range'], 0.5
    )

    f['rsi14'] = np.array([compute_rsi(p[i]) for i in range(n)])

    t  = np.arange(40)
    tn = (t - t.mean()) / t.std()
    lp = np.log(p)
    f['trend_slope'] = (lp * tn).mean(axis=1) - lp.mean(axis=1) * tn.mean()

    f['ret_2nd_half']   = np.log(p[:, -1] / p[:, 19])
    f['ret_1st_half']   = np.log(p[:, 19] / p[:, 0])
    f['momentum_accel'] = f['ret_2nd_half'] - f['ret_1st_half']

    out = pd.DataFrame(f, index=df.index)
    out.replace([np.inf, -np.inf], np.nan, inplace=True)
    out.fillna(out.median(), inplace=True)
    return out


print('Building features for train...')
X_all  = engineer_features(train)
print('Building features for test...')
X_test = engineer_features(test)
y_all  = train['p50']

print(f'Feature matrix: {X_all.shape}')
print(f'Any NaN: {X_all.isna().any().any()}')
X_all.head(3)

## Train / Validation Split

In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_all, y_all, test_size=0.2, random_state=RANDOM_STATE
)

p40_val = train.loc[X_val.index, 'p40']
p50_val = train.loc[X_val.index, 'p50']

print(f'Train: {X_tr.shape}  |  Val: {X_val.shape}')

## Train Models

We train 4 gradient boosting models and average their predictions.  
Each model uses early stopping — it trains until the validation error stops improving, so we don't need to guess the right number of trees.

In [ ]:
# LightGBM
lgbm_params = dict(
    objective='regression', metric='rmse', learning_rate=0.05,
    num_leaves=63, min_child_samples=20, feature_fraction=0.8,
    bagging_fraction=0.8, bagging_freq=5, reg_alpha=0.1, reg_lambda=1.0,
    n_estimators=1000, random_state=RANDOM_STATE, verbose=-1
)
lgbm_model = lgb.LGBMRegressor(**lgbm_params)
lgbm_model.fit(
    X_tr, y_tr, eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(200)]
)
lgbm_preds = lgbm_model.predict(X_val)
print(f'LightGBM  RMSE: {mean_squared_error(y_val, lgbm_preds)**0.5:.4f}  (stopped at iter {lgbm_model.best_iteration_})')

In [ ]:
# XGBoost
xgb_model = xgb.XGBRegressor(
    objective='reg:squarederror', learning_rate=0.05, max_depth=5,
    n_estimators=1000, subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.0, random_state=RANDOM_STATE,
    verbosity=0, early_stopping_rounds=50
)
xgb_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=200)
xgb_preds = xgb_model.predict(X_val)
print(f'XGBoost   RMSE: {mean_squared_error(y_val, xgb_preds)**0.5:.4f}  (stopped at iter {xgb_model.best_iteration})')

In [ ]:
# CatBoost
cat_model = CatBoostRegressor(
    iterations=1000, learning_rate=0.05, depth=6, l2_leaf_reg=3,
    random_seed=RANDOM_STATE, eval_metric='RMSE',
    early_stopping_rounds=50, verbose=200
)
cat_model.fit(X_tr, y_tr, eval_set=(X_val, y_val))
cat_preds = cat_model.predict(X_val)
print(f'CatBoost  RMSE: {mean_squared_error(y_val, cat_preds)**0.5:.4f}  (stopped at iter {cat_model.best_iteration_})')

In [ ]:
# HistGradientBoosting (sklearn built-in, no extra install needed)
hgb_model = HistGradientBoostingRegressor(
    max_iter=500, learning_rate=0.05, max_leaf_nodes=63,
    min_samples_leaf=20, l2_regularization=1.0,
    random_state=RANDOM_STATE, early_stopping=True,
    validation_fraction=0.1, n_iter_no_change=30
)
hgb_model.fit(X_tr, y_tr)
hgb_preds = hgb_model.predict(X_val)
print(f'HistGBM   RMSE: {mean_squared_error(y_val, hgb_preds)**0.5:.4f}  (stopped at iter {hgb_model.n_iter_})')

In [ ]:
# Average the 4 model predictions
ensemble_preds = (lgbm_preds + xgb_preds + cat_preds + hgb_preds) / 4

results = pd.DataFrame({
    'Model':  ['LightGBM', 'XGBoost', 'CatBoost', 'HistGBM', 'Ensemble'],
    'RMSE':   [
        mean_squared_error(y_val, lgbm_preds) ** 0.5,
        mean_squared_error(y_val, xgb_preds)  ** 0.5,
        mean_squared_error(y_val, cat_preds)  ** 0.5,
        mean_squared_error(y_val, hgb_preds)  ** 0.5,
        mean_squared_error(y_val, ensemble_preds) ** 0.5,
    ],
    'MAE': [
        mean_absolute_error(y_val, lgbm_preds),
        mean_absolute_error(y_val, xgb_preds),
        mean_absolute_error(y_val, cat_preds),
        mean_absolute_error(y_val, hgb_preds),
        mean_absolute_error(y_val, ensemble_preds),
    ]
}).set_index('Model').round(4)

display(results)

## Evaluate Competition R on Validation Set

Expected gain for each stock = p40 - predicted p50.  
We rank by that, take the top K stocks, and compute R.

In [ ]:
def competition_score(p40, p50, preds, max_sell):
    expected_gain = p40.values - preds
    ranked = np.argsort(expected_gain)[::-1]
    sell = np.zeros(len(preds), dtype=int)
    sell[ranked[:max_sell]] = 1
    R = float((sell * (p40.values - p50.values)).sum())
    n_prof = int((sell * (p40.values > p50.values)).sum())
    print(f'Stocks sold:           {sell.sum()}')
    print(f'Actually profitable:   {n_prof} ({n_prof/sell.sum():.1%})')
    print(f'Competition R:         {R:.2f}')
    return R


K_val = int(0.4 * len(X_val))
R_val = competition_score(p40_val, p50_val, ensemble_preds, K_val)
print(f'\nEstimated test R (~5x): {R_val * 5:.0f}')

## Feature Importance

In [ ]:
feat_imp = pd.Series(
    lgbm_model.feature_importances_,
    index=X_all.columns
).sort_values(ascending=False)

plt.figure(figsize=(9, 6))
feat_imp.head(15).plot(kind='barh')
plt.gca().invert_yaxis()
plt.title('LightGBM Feature Importance (top 15)')
plt.tight_layout()
plt.show()

## Retrain on Full Training Data

Now that we know it works, we train each model on all 10,000 training rows using the number of iterations found by early stopping above.

In [ ]:
print(f'Best iterations — LGB: {lgbm_model.best_iteration_}  XGB: {xgb_model.best_iteration}  CAT: {cat_model.best_iteration_}  HGB: {hgb_model.n_iter_}')

lgbm_final = lgb.LGBMRegressor(**{**lgbm_params, 'n_estimators': lgbm_model.best_iteration_})
lgbm_final.fit(X_all, y_all)
print('LightGBM retrained')

xgb_final = xgb.XGBRegressor(
    objective='reg:squarederror', learning_rate=0.05, max_depth=5,
    n_estimators=xgb_model.best_iteration, subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.0, random_state=RANDOM_STATE, verbosity=0
)
xgb_final.fit(X_all, y_all)
print('XGBoost retrained')

cat_final = CatBoostRegressor(
    iterations=cat_model.best_iteration_, learning_rate=0.05, depth=6,
    l2_leaf_reg=3, random_seed=RANDOM_STATE, verbose=False
)
cat_final.fit(X_all, y_all)
print('CatBoost retrained')

hgb_final = HistGradientBoostingRegressor(
    max_iter=hgb_model.n_iter_, learning_rate=0.05, max_leaf_nodes=63,
    min_samples_leaf=20, l2_regularization=1.0, random_state=RANDOM_STATE
)
hgb_final.fit(X_all, y_all)
print('HistGBM retrained')

## Generate Test Predictions & Submission

In [ ]:
test_p50_pred = (
    lgbm_final.predict(X_test)
    + xgb_final.predict(X_test)
    + cat_final.predict(X_test)
    + hgb_final.predict(X_test)
) / 4

p40_test     = test['p40'].values
expected_gain = p40_test - test_p50_pred

ranked = np.argsort(expected_gain)[::-1]
sell_test = np.zeros(len(test), dtype=int)
sell_test[ranked[:MAX_SELL]] = 1

print(f'Total sells: {sell_test.sum()}')
print(f'Constraint satisfied (<=4000): {sell_test.sum() <= MAX_SELL}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(expected_gain, bins=80, color='steelblue', alpha=0.8)
axes[0].axvline(0, color='red', ls='--', lw=1.5, label='breakeven')
cutoff = expected_gain[ranked[MAX_SELL - 1]]
axes[0].axvline(cutoff, color='orange', ls='--', lw=1.5, label=f'sell cutoff = {cutoff:.2f}')
axes[0].set_title('Expected gain distribution (test set)')
axes[0].legend()

axes[1].hist(test_p50_pred, bins=80, color='darkorange', alpha=0.8)
axes[1].set_title('Predicted p50 distribution (test set)')

plt.tight_layout()
plt.show()

In [ ]:
submission = pd.DataFrame({'ID': test.index, 'sell': sell_test})
submission.to_csv('submission.csv', index=False)

print('submission.csv saved')
print(submission['sell'].value_counts().rename({0: 'hold', 1: 'sell'}))
submission.head(10)